The main objective is to analyze the event log data and find the decision places via token replay using alignments beforehand.
But using now our bpmn because of the a lot of skip and tau in the event log that can mislead us.

In [ ]:
import sys
import os
import pm4py
import pandas as pd
import numpy as np
import json
import graphviz
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.petri_net.semantics import enabled_transitions, execute
from c45_tree import C45DecisionTree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

In [ ]:
current_dir = os.getcwd()
sim_core_path = os.path.abspath(os.path.join(current_dir, '..', 'sim_core'))
if sim_core_path not in sys.path:
    sys.path.append(sim_core_path)
try:
    from pn_model import PNModel, wrap_net
    from bpmn_io import read_bpmn
except ImportError as e:
    print(f"Error: {e}")

In [41]:
bpmn_path = "../data/process_model.bpmn"
log_path = "../data/BPI Challenge 2017.xes.gz"

In [ ]:
log = pm4py.read_xes(log_path)

parsing log, completed traces :: 100%|██████████| 31509/31509 [00:22<00:00, 1372.26it/s]


In [44]:
bpmn_graph = pm4py.read_bpmn(bpmn_path)

In [45]:
log_activities = set(log['concept:name'].unique())
bpmn_labels = set()
for node in bpmn_graph.get_nodes():
    if node.get_name() and node.get_name().strip() != "":
        bpmn_labels.add(node.get_name().strip())

missing_in_bpmn = log_activities - bpmn_labels
missing_in_bpmn

{'O_Sent (online only)',
 'W_Assess potential fraud',
 'W_Personal Loan collection',
 'W_Shortened completion '}

It is normal to get such as average fitness because as we discovered: we didnt model these activites in our bpmn 'O_Sent (online only)',
 'W_Assess potential fraud',
 'W_Personal Loan collection',
 'W_Shortened completion '
Otherwise, the data would be too noisy

Decided to the training with event log 

In [ ]:
df = pm4py.convert_to_dataframe(log)
print(f"Loaded {len(df)} events.")
df_log = pm4py.convert_to_dataframe(log)
df_log['time:timestamp'] = pd.to_datetime(df_log['time:timestamp'])
df_log = df_log.sort_values(['case:concept:name', 'time:timestamp'])

Loaded 1202267 events.


## Case Attributes

In [ ]:

#  Build case attribute lookup from event log
case_attr_cols = ["case:concept:name", "case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]
available_cols = [c for c in case_attr_cols if c in df_log.columns]
case_attrs = df_log.groupby("case:concept:name").first()[available_cols[1:]].reset_index()
case_attrs.columns = ["case_id", "loan_goal", "application_type", "requested_amount"]
case_attrs["loan_goal"] = case_attrs["loan_goal"].fillna("Unknown").astype(str)
case_attrs["application_type"] = case_attrs["application_type"].fillna("Unknown").astype(str)
case_attrs["requested_amount"] = pd.to_numeric(case_attrs["requested_amount"], errors="coerce").fillna(0)

# Amount category
case_attrs["amount_category"] = pd.cut(
    case_attrs["requested_amount"],
    bins=[0, 5000, 10000, 20000, 50000, float("inf")],
    labels=["very_low", "low", "medium", "high", "very_high"]
).astype(str)

case_lookup = case_attrs.set_index("case_id").to_dict("index")
print(f"Case attributes loaded for {len(case_lookup)} cases.")
print(f"LoanGoal dist: {case_attrs['loan_goal'].value_counts().to_dict()}")
print(f"Amount dist:   {case_attrs['amount_category'].value_counts().to_dict()}")

# Build CreditScore lookup per offer event
# CreditScore is on O_Create Offer events — extract max per case
offer_events = df_log[df_log["concept:name"] == "O_Create Offer"].copy()
credit_cols = [c for c in ["CreditScore", "OfferedAmount"] if c in offer_events.columns]
if credit_cols:
    case_credit = offer_events.groupby("case:concept:name")[credit_cols].max().reset_index()
    case_credit.columns = ["case_id"] + credit_cols
    for _, row in case_credit.iterrows():
        cid = row["case_id"]
        if cid in case_lookup:
            if "CreditScore" in credit_cols:
                cs = row.get("CreditScore", None)
                if pd.notna(cs):
                    cs = float(cs)
                    if cs > 800: case_lookup[cid]["credit_score_bin"] = "excellent"
                    elif cs > 600: case_lookup[cid]["credit_score_bin"] = "good"
                    elif cs > 400: case_lookup[cid]["credit_score_bin"] = "fair"
                    else: case_lookup[cid]["credit_score_bin"] = "poor"
    print("CreditScore extracted from offer events.")
else:
    print("No CreditScore/OfferedAmount columns found.")

In [52]:
tree = inductive_miner.apply(log)

net, initial_marking, final_marking = pt_converter.apply(tree)

In [53]:
replayed_log = token_replay.apply(log, net, initial_marking, final_marking)

replaying log with TBR, completed traces :: 100%|██████████| 15930/15930 [00:38<00:00, 416.05it/s]


In [54]:
event_log = pm4py.convert_to_event_log(log)

(skip_37, None),(skip_38, None),(tauSplit_40, None),(skip_48, None), etc. are the silent transitions that probably lead to a XOR

In [ ]:
decision_training_data = []

for trace_idx, trace in enumerate(event_log):
    replay_result = replayed_log[trace_idx]
    fired_transitions = replay_result['activated_transitions']
    event_idx = 0

    case_id = trace.attributes.get('concept:name')
    case_start_time = trace[0]['time:timestamp']
    current_marking = initial_marking.copy()

    # case-level attributes from lookup
    c_info = case_lookup.get(case_id, {})
    loan_goal = c_info.get("loan_goal", "Unknown")
    app_type = c_info.get("application_type", "Unknown")
    amount_cat = c_info.get("amount_category", "medium")
    credit_bin = c_info.get("credit_score_bin", "unknown")

    # per-case tracking
    activity_counts = {}
    offer_count = 0
    rejection_count = 0
    accepted_offer = False

    for t in fired_transitions:
        enabled = enabled_transitions(net, current_marking)

        if len(enabled) > 1:
            if event_idx > 0 and event_idx < len(trace):
                prev_event = trace[event_idx - 1]
                curr_event = trace[event_idx]

                decision_label = t.label if t.label else t.name
                prev_act = prev_event['concept:name']

                features = {
                    "case_id": case_id,
                    "prev_activity": prev_act,
                    "last_resource": prev_event.get('org:resource', 'System'),
                    "last_duration": (curr_event['time:timestamp'] - prev_event['time:timestamp']).total_seconds(),
                    "case_duration_hours": (curr_event['time:timestamp'] - case_start_time).total_seconds() / 3600,
                    "target_decision": decision_label,
                    "loan_goal": loan_goal,
                    "application_type": app_type,
                    "amount_category": amount_cat,
                    "credit_score_bin": credit_bin,
                    "offer_count": offer_count,
                    "has_rejection": 1 if rejection_count > 0 else 0,
                    "has_accepted_offer": 1 if accepted_offer else 0,
                    "is_repeated": str(activity_counts.get(prev_act, 0) > 1),
                }
                decision_training_data.append(features)

        # State update
        current_marking = execute(t, net, current_marking)

        if t.label is not None:
            act_name = t.label
            activity_counts[act_name] = activity_counts.get(act_name, 0) + 1

            # track business events
            if act_name in ("O_Created", "O_Create Offer"):
                offer_count += 1
            if act_name in ("O_Refused", "O_Cancelled", "A_Denied", "A_Cancelled"):
                rejection_count += 1
            if act_name == "O_Accepted":
                accepted_offer = True

            event_idx += 1

df_xor_decisions = pd.DataFrame(decision_training_data)

print(f"Toplam Karar Satiri: {len(df_xor_decisions)}")
print(f"Benzersiz Önceki Aktiviteler: {df_xor_decisions['prev_activity'].nunique()}")
print(f"Benzersiz Hedef Kararlar: {df_xor_decisions['target_decision'].nunique()}")

Toplam Karar Satırı: 2819057
Benzersiz Önceki Aktiviteler: 26
Benzersiz Hedef Kararlar: 83


In [56]:
df_xor_decisions.head()

,case_id,prev_activity,last_resource,last_duration,case_duration_hours,target_decision
0,Application_652823628,A_Create Application,User_1,0.048,0.000013,A_Submitted
1,Application_652823628,A_Submitted,User_1,0.422,0.000131,init_loop_11
2,Application_652823628,A_Submitted,User_1,0.422,0.000131,W_Handle leads
3,Application_652823628,W_Handle leads,User_1,80.618,0.022524,skip_13
4,Application_652823628,W_Handle leads,User_1,80.618,0.022524,W_Handle leads


In [57]:
# Target Activities with silent transitions
print(df_xor_decisions['target_decision'].value_counts().head(30))

target_decision
W_Call after offers         179625
skip_25                     148742
W_Complete application      147848
skip_65                     134654
W_Validate application      117067
skip_72                     116345
skip_35                      88572
skip_70                      87475
skip_19                      84588
tauSplit_40                  77274
W_Call incomplete files      71866
tauJoin_41                   69808
skip_36                      69313
skip_48                      64408
skip_39                      63326
tauSplit_20                  63003
skip_42                      62594
init_loop_29                 60428
skip_45                      58850
init_loop_33                 47754
W_Handle leads               47208
skip_71                      47179
skip_37                      41980
skip_38                      41936
O_Create Offer               40923
O_Created                    40724
skip_31                      39376
O_Sent (mail and online)     37983
skip

In [ ]:
matrix_view = pd.crosstab(df_xor_decisions['target_decision'], df_xor_decisions['prev_activity'])

plt.figure(figsize=(20, 15))
sns.heatmap(matrix_view, cmap="YlGnBu", annot=False)
plt.title("Decision points vs Prev Activity")
plt.xlabel("Prev Activity")
plt.ylabel("Target Decision")
plt.show()

## Cleaning

In [ ]:
df_clean = df_xor_decisions[
    (df_xor_decisions['target_decision'].notna()) &
    (df_xor_decisions['target_decision'] != "") &
    (~df_xor_decisions['target_decision'].str.contains('tau|skip|init|loop', case=False))
].copy()

In [ ]:
counts = df_clean['target_decision'].value_counts()
significant_targets = counts[counts > 50].index
df_clean = df_clean[df_clean['target_decision'].isin(significant_targets)]
print(f"After filtering: {len(df_clean):,} rows, {df_clean['target_decision'].nunique()} targets")
print(df_clean['target_decision'].value_counts())

# Binning

In [ ]:
df_clean['duration_bin'], duration_bins = pd.qcut(
    df_clean['last_duration'], q=5,
    labels=['VeryShort', 'Short', 'Medium', 'Long', 'VeryLong'],
    duplicates='drop', retbins=True
)

df_clean['case_age_category'], case_age_bins = pd.qcut(
    df_clean['case_duration_hours'], q=4,
    labels=['Very_New', 'New', 'Old', 'Delayed'],
    duplicates='drop', retbins=True
)

# NEW: offer count category
df_clean['offer_category'] = df_clean['offer_count'].apply(
    lambda x: 'none' if x == 0 else ('single' if x == 1 else 'multiple')
)

print("Feature distributions:")
for col in ['loan_goal', 'amount_category', 'credit_score_bin', 'offer_category', 'has_rejection']:
    print(f"  {col}: {df_clean[col].value_counts().to_dict()}")

## Train the tree C4.5

In [ ]:
features = [
    "prev_activity",        
    "duration_bin",         
    "case_age_category",    
    "loan_goal",            
    "application_type",     
    "amount_category",      
    "credit_score_bin",     
    "offer_category",       
    "has_rejection",        
    "has_accepted_offer",   
    "is_repeated",          
]

attribute_types = {f: "nominal" for f in features}

X = df_clean[features].astype(str)
y = df_clean["target_decision"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

final_tree = C45DecisionTree(
    attribute_types=attribute_types,
    max_depth=30,
    min_samples_split=10
)
final_tree.fit(X_train, y_train)

In [ ]:
importance = final_tree.get_feature_importance()
print("\nFeature Importance:")
for feat, score in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat}: {score:.4f}")

In [ ]:
dots = final_tree.export_graphviz()
graph = graphviz.Source(dots)
graph

In [ ]:
sim_package = {
    "model": final_tree,
    "bins": {
        "duration_bin": duration_bins.tolist() if hasattr(duration_bins, 'tolist') else duration_bins,
        "case_age_category": case_age_bins.tolist() if hasattr(case_age_bins, 'tolist') else case_age_bins
    },
    # NEW: save feature list so router knows what to send
    "features": features,
}

with open("simulation_brain.pkl", "wb") as f:
    pickle.dump(sim_package, f)

print(f"Model saved. Features: {features}")